# [ISR06 미션 03] 정보 출처별 상충 정보 식별 및 교차검증(Cross-validation) 세부 평가 기준 수립 — 교수자용 해설판 (Workbook)

- **산출물 경로**: `analysis/isr06_uncertainty_assessment.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·위치·수집 수단·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 3. 엇갈린 보고와 확인할 내용을 정리한 문서 (필수)

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「엇갈린 보고와 확인할 내용을 정리한 문서」 초안을 만듭니다.

보고끼리 맞지 않는다고 바로 하나를 틀렸다고 버리면 중요한 차이를 놓칠 수 있습니다.

차이가 생긴 지점과 더 확인할 내용을 분리하면 신중한 문서를 만들 수 있습니다.

준비할 자료는 다음과 같습니다: 보고 K1~K4, 보고 접수대장의 관련 행, 미션 2 기준표. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

### 상황과 자료

네 쌍의 보고 K1~K4에서 시각·위치·수량 등이 서로 다릅니다.

A팀은 확정된 내용과 아직 모르는 내용을 B팀에 구분해 전달해야 합니다.

**준비할 자료**: 보고 K1~K4, 보고 접수대장의 관련 행, 미션 2 기준표

**뜻 알아두기**: 상충은 두 기록이 서로 맞지 않는 상태입니다.

관측 시각은 내용을 본 때, 접수 시각은 보고가 도착한 때입니다.

K3-B=C3 표시는 접수 시각을 연결하는 것이며 두 본문 전체가 같다는 뜻은 아닙니다.

In [ ]:
# [환경 준비] 미션 3 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/출처_이력대장.csv': "[교육용가상자료] 출처 이력대장 — 가상 조직 A팀 출처 풀. 등록일·보고 건수는 D일 기준이며 실제 수집 수단·인물과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n출처ID,출처유형,출처설명,등록일,과거보고건수,확인된적중건수,적중률_퍼센트,검증이력상태,전달경로,평균전달지연,비고\nH-1,인터뷰,현지 협조자(나-3 교량 북쪽 마을 거주),D-210일,8,6,87.5,검증 이력 있음,연락관 경유 구두 보고,약 6시간,관측 시각을 보고에 적지 않는 습관이 있음\nH-2,인터뷰,신규 협조자(H-1의 소개),D-1일,0,0,해당 없음,검증 이력 없음,H-1 경유 전언,불명,첫 보고가 H-1에게 들은 내용의 전달\nH-3,인터뷰,다-2 도로변 협조자,D-95일,3,2,66.7,검증 이력 있음,직접 통화,약 40분,시각을 '○○시경'으로만 적음\nS-1,통신기록,가상 통신 요약 판독팀,상시 운용,40,36,90.0,검증 이력 있음,자동 요약 후 판독관 확인,약 1시간,통신 내용은 포함하지 않고 통신량·구역만 요약\nI-1,사진자료,위성영상 판독팀,상시 운용,25,23,92.0,검증 이력 있음,판독 의견서,약 20분,영상 시각·해상도 등급 기재. 구름·야간에 공백\nI-2,사진자료,드론 영상 운용조(다-2 도로 전담),상시 운용,12,11,91.7,검증 이력 있음,실시간 전송,약 5분,배터리 교체마다 20분 공백\nO-1,공개자료,공개 게시 계정(신원 불명),등록 없음,0,0,해당 없음,검증 불가,공개망 열람,해당 없음,이전 게시물 사진 재사용 전력 1회 의심\nO-2,공개자료,공개 게시 계정(O-1 인용 계정군),등록 없음,0,0,해당 없음,검증 불가,공개망 열람,해당 없음,O-1 사진을 인용해 게시\n", '02_데이터셋/보고_접수대장.csv': "[교육용가상자료] 보고 접수대장 — 가상 조직 A팀 D일 접수분(사전 접수분 포함). '(미기재)'는 보고에 그 값이 없었다는 뜻이며 추정해 채우지 않는다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n보고ID,출처ID,출처유형,관측시각,접수시각,격자,내용요약,재인용원출처,비고,쓰이는미션\nP2,H-2,인터뷰,(미기재),D-3일 09:10,가-4,가-4 일대 낯선 차량 2대 목격,,사전 접수분,참고\nP1,H-1,인터뷰,D-2일 14:00,D-2일 20:30,나-3,교량 인근 차량 통행 평시 수준,,사전 접수분,참고\nC1,H-1,인터뷰,(미기재),07:40,나-3 북단,군용 차량 10여 대 정차,,연락관 추정 전달 지연 약 6시간,M1·M2(R1)·M3(K1-A)·M5\nC2,S-1,통신기록,05:00~07:00(수집 구간),07:55,가-4,통신량 평소의 3배,,통신 내용 불명,M1·M2(R4)·M3(K4-A)\nC5,H-2,인터뷰,(해당 없음),08:30,(미기재),H-1 말로는 교량 쪽이 위험하다고 함,H-1,2차 전달,M1·M2(R5)\nC3,I-1,사진자료,09:10,09:30,나-3 일대,차량형 물체 8개 판독(구름 일부),,해상도 보통,M1·M2(R2)·M3(K1-B·K3-B)·M5\nC4,O-1,공개자료,(미확인),10:20,(미확인),교량 앞 군용 행렬 사진 1장,,촬영 시각·장소 미확인·재사용 의심,M1·M2(R3)·M3(K3-A)·M5\nO2-1,O-2,공개자료,(해당 없음),10:45,(미확인),O-1 사진을 인용한 게시물,O-1,재인용,M7 참고\nK4B,H-2,인터뷰,(미기재),10:50,가-4,해당 조직이 정비 중이라고 함,(미기재),전언 여부 미기재,M3(K4-B)\nK2A,I-2,사진자료,11:00,11:05,다-2 도로,이동 차량 없음,,배터리 교체 공백 10:35~10:55 직후 촬영,M3(K2-A)·M5·M8(A)\nK2B,H-3,인터뷰,11:00경,11:40,다-2 도로,차량 행렬 통과,,'경' 표기,M3(K2-B)·M5·M8(A)\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답이나 학습자 수행물이 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n가중치: 지지 +1, 반대 -1, 미확인 ?의 감점계수 1. 점수=지지합−반대합−?개수. 여기서 반대합은 음수 가중치를 합친 값의 절댓값입니다. 반대 근거와 미확인 항목은 점수를 낮춥니다. 규칙1: 반대 근거 또는 미해소 상충이 하나라도 있으면 보류. 규칙2: 점수3 미만이면 추가 수집 요청. 규칙3: 독립 출처 유형 N은 미션 2에서 정한 값을 그대로 사용(미정이면2). 규칙4: 사람의 최종 판단·운영 규칙 우선, 자동 승인 금지. 미션 4의12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다. 미션 5에서는 독립 출처 개수·상충을 실제 자료로 대조한다.\n\n## 미션 6 대체 빈 서식\n\n| 보고ID | 출처ID | 사실 | 판단 | 조치 | 미확인 | 관측시각 | 접수시각 | 근거 |\n|---|---|---|---|---|---|---|---|---|\n| | | | | | | | | |\n\n미션 9의 요약 차트는 미션 5·미션 7 결과를 이 빈 서식에 정리한다. 미션 6 자체의 보고3종 전사·8출처 대조 수행은 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 원문 AAR A·B를 미션 9에서 읽어 B1~B5와 함께7건에 두 기준을 적용한다. 이전 판단이 없는 A·B는 미션 9에서 이전 기준을 재적용하고 과거 수행값처럼 쓰지 않는다. 같은 결과는 유지 이유를 기록한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 아래 수정은 원문 사실 변경이나 현재 실습에 적용할 변경 사항입니다.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| 정확성 척도 | 정확성 ( )단계 이하 | 숫자 코드 n 이상(n~6) | 낮은 신뢰 수준과 숫자 방향 구별 | 미션 2 서식3 및 후속 규칙 |\n| 접수·관측 시각 | 관측 미기재이면 접수 차이 계산 불가 | 접수 차이와 관측 차이 각각 계산 | 서로 다른 시간열 | 미션 3 K1 접수 차이110분, 관측 차이 계산 불가 |\n| 개인 독립 판정 | 조원 1~4 열 / 4건 이상 일치 | 학습자·AI1·AI2·해당 없음 / 보고5건 중4건 | 평가자 수와 보고 건수 구분 | 미션 2 서식2 |\n| 잠정 행정판단 | 이 미션에서는 결론 금지 | 사실 단정 금지, 잠정 행정판단 필수 | 판단 종류 구분 | 미션 3 서식1·확인 방법 |\n| 미션 7 카드 관측시각 | D5 같은 시각, 기준 없음 | D1·D5 관측09:10; C3와 별도 카드 실험 | 수량 상충 판단 근거 제공; 10여 대/3개 유지 | 미션 7 카드 실험; 경과시간 분:초는 별도 |\n| 학습시간 | 원문 소요시간/배정시간 이중 기준 | 학습 계획은 원문 시간 기준 | 원문 합계27시간 유지; 현실 시간 미실측 | 전 미션 시간 계획 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '02_데이터셋/isr06_m03_sample_2.md': "# ISR06 미션 3 실습 샘플\n\n> `02_데이터셋/ISR06_실습자료_수정실습본.md`의 미션 3 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 3. 정보 출처별 상충 정보 식별 및 교차검증(Cross-validation) 세부 평가 기준 수립**\n\n가. 제공 자료 1 - 가상 상충 보고 쌍 4건 [교육용가상자료]\n\n| **쌍** | **보고 A** | **보고 B** | **상충 지점** |\n| --- | --- | --- | --- |\n| **K1** | H-1: 교량 북단에 차량 10여 대 정차(오전) | I-1: 같은 구역 차량형 물체 8개(09:10) | 수량(10여 대와 8개) |\n| **K2** | I-2 드론: 다-2 도로에 이동 차량 없음(11:00) | H-3: 같은 도로에 차량 행렬 통과(11:00경) | 존재 여부 |\n| **K3** | O-1 게시물: 나-3 교량 앞 군용 행렬(사진) | I-1: 같은 시간대 교량 일대에 사진과 같은 규모 없음 | 시각·위치 |\n| **K4** | S-1: 가-4 통신량 증가를 '이동 준비' 가능성으로 판독 | H-2: 같은 조직이 '정비 중'이라고 보고 | 의미(주체의 의도) |\n\n가. 제공 자료 2 - 정보 공백 예시 3개(나머지 5개 이상은 조별로 찾는다)\n\n| **정보 공백 요인** | **발생 상황** | **대체 수집 수단(예)** |\n| --- | --- | --- |\n| **관측 시각 미기재** | 협조자 보고에 관측 시각이 없어 이동 시점 불명 | 협조자 재접촉으로 시각 확인 요청, 영상 시각과 대조 |\n| **영상 공백** | 구름·야간·드론 배터리 교체로 일정 구간 영상 없음 | 다른 시각 영상 확보, 지상 관측 보완 |\n| **단일 언어·단일 지역 편중** | 공개출처가 한쪽 매체에만 의존 | 다른 언어·매체 보고 대조 |\n\n나. 작성 서식 1 - 상충 분류와 원인 후보\n\n| **쌍** | **상충 유형(시각·위치·수량·주체·의미)** | **원인 후보(출처 오류, 시차, 허위, 재인용)** | **확인 방법** | **결론(승인·보류·추가 수집)** |\n| --- | --- | --- | --- | --- |\n| **K1** | | | | |\n| **K2** | | | | |\n| **K3** | | | | |\n| **K4** | | | | |\n\n나. 작성 서식 2 - 정보 공백·불확실성 영향도 매트릭스(8개 이상)\n\n| **번호** | **정보 공백 요인** | **영향도(높음·중간·낮음)** | **영향을 받는 판단** | **대체 수집 수단** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n| **7** | | | | |\n| **8** | | | | |\n| **9** | | | | |\n| **10** | | | | |\n\n나. 작성 서식 3 - 확신 표현 규칙과 판단의 한계: 근거가 한 출처뿐일 때 결론에 쓸 수 있는 표현은 ( ), 쓰면 안 되는 표현은 ( ). 이 평가로 판단할 수 없는 경우 3가지: 1) ( ) 2) ( ) 3) ( )\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 상충 보고 4건 모두에 상충 유형과 원인 후보를 2개 이상 적었다 | ( ) |\n| 정보 공백 요인을 8개 이상 찾고 각각의 영향도를 매겼다 | ( ) |\n| 정보 공백마다 대체 수집 수단을 1개 이상 적었다 | ( ) |\n| 과도한 확신을 막는 표현 규칙을 정했다 | ( ) |\n| 판단의 한계 3가지를 적었다 | ( ) |\n", 'analysis/isr06_uncertainty_assessment.md': '# 미션 3 analysis/isr06_uncertainty_assessment.md(상충 분류표, 정보 공백·불확실성 영향도 매트릭스, 판단 한계 문서) (필수)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}

for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 3 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')

### 할 일

1. 두 보고의 문장과 관측·접수 시각을 서로 다른 칸에 적습니다.

2. AI에게 차이의 유형·원인 후보·확인 방법을 담은 문서 초안을 요청합니다.

3. 원인 후보를 자료와 대조하고 근거 없는 원인을 사실처럼 쓴 곳을 고칩니다.

4. 공백 8개 이상과 판단 한계 3가지를 적고 확인 담당자와 방법을 연결합니다.

5. 상충 정리표·공백 표·확신 표현 규칙을 한 문서로 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

### 수행 단계 (guideSteps 요약)
1. **두 보고의 문장과 관측·접수 시각을 서로 다른 칸에 적습니다**
2. **AI와 함께 해보기 — 엇갈린 보고와 확인할 내용을 정리한 문서 초안**
3. **원인 후보를 자료와 대조하고 근거 없는 원인을 사실처럼 쓴 곳을 고칩니다**
4. **공백 8개 이상과 판단 한계 3가지를 적고 확인 담당자와 방법을 연결합니다**
5. **상충 정리표·공백 표·확신 표현 규칙을 한 문서로 저장합니다**

---

가. 제공 자료 1 - 가상 상충 보고 쌍 4건 [교육용가상자료]

| **쌍** | **보고 A** | **보고 B** | **상충 지점** |
| --- | --- | --- | --- |
| **K1** | H-1: 교량 북단에 차량 10여 대 정차(오전) | I-1: 같은 구역 차량형 물체 8개(09:10) | 수량(10여 대와 8개) |
| **K2** | I-2 드론: 다-2 도로에 이동 차량 없음(11:00) | H-3: 같은 도로에 차량 행렬 통과(11:00경) | 존재 여부 |
| **K3** | O-1 게시물: 나-3 교량 앞 군용 행렬(사진) | I-1: 같은 시간대 교량 일대에 사진과 같은 규모 없음 | 시각·위치 |
| **K4** | S-1: 가-4 통신량 증가를 '이동 준비' 가능성으로 판독 | H-2: 같은 조직이 '정비 중'이라고 보고 | 의미(주체의 의도) |

가. 제공 자료 2 - 정보 공백 예시 3개(나머지 5개 이상은 조별로 찾는다)

| **정보 공백 요인** | **발생 상황** | **대체 수집 수단(예)** |
| --- | --- | --- |
| **관측 시각 미기재** | 협조자 보고에 관측 시각이 없어 이동 시점 불명 | 협조자 재접촉으로 시각 확인 요청, 영상 시각과 대조 |
| **영상 공백** | 구름·야간·드론 배터리 교체로 일정 구간 영상 없음 | 다른 시각 영상 확보, 지상 관측 보완 |
| **단일 언어·단일 지역 편중** | 공개출처가 한쪽 매체에만 의존 | 다른 언어·매체 보고 대조 |

### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「엇갈린 보고와 확인할 내용을 정리한 문서」의 초안을 만들어 줘.
자료: 보고 K1~K4, 보고 접수대장의 관련 행, 미션 2 기준표.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 사례 ID | 서로 다른 내용 | 원인 후보 2개 이상 | 확인 방법 | 잠정 결론. 공백 | 영향 | 대체 확인 수단.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식 원본 (참고)

나. 작성 서식 1 - 상충 분류와 원인 후보

| **쌍** | **상충 유형(시각·위치·수량·주체·의미)** | **원인 후보(출처 오류, 시차, 허위, 재인용)** | **확인 방법** | **결론(승인·보류·추가 수집)** |
| --- | --- | --- | --- | --- |
| **K1** | | | | |
| **K2** | | | | |
| **K3** | | | | |
| **K4** | | | | |

나. 작성 서식 2 - 정보 공백·불확실성 영향도 매트릭스(8개 이상)

| **번호** | **정보 공백 요인** | **영향도(높음·중간·낮음)** | **영향을 받는 판단** | **대체 수집 수단** |
| --- | --- | --- | --- | --- |
| **1** | | | | |
| **2** | | | | |
| **3** | | | | |
| **4** | | | | |
| **5** | | | | |
| **6** | | | | |
| **7** | | | | |
| **8** | | | | |
| **9** | | | | |
| **10** | | | | |

나. 작성 서식 3 - 확신 표현 규칙과 판단의 한계: 근거가 한 출처뿐일 때 결론에 쓸 수 있는 표현은 ( ), 쓰면 안 되는 표현은 ( ). 이 평가로 판단할 수 없는 경우 3가지: 1) ( ) 2) ( ) 3) ( )

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| 상충 보고 4건 모두에 상충 유형과 원인 후보를 2개 이상 적었다 | ( ) |
| 정보 공백 요인을 8개 이상 찾고 각각의 영향도를 매겼다 | ( ) |
| 정보 공백마다 대체 수집 수단을 1개 이상 적었다 | ( ) |
| 과도한 확신을 막는 표현 규칙을 정했다 | ( ) |
| 판단의 한계 3가지를 적었다 | ( ) |

## [교수자용 해설 및 모범 답안]

### 1. 지도 포인트 (정오표 반영 사항)
- **시각 차이 분리 계산**: 관측 시각이 `(미기재)`여도 **접수 시각 차이는 분 단위로 계산**합니다 (`K1` 접수 차 = `110분`, 관측 차 = `계산 불가`).
- **잠정 행정판단 3종 준수**: 결론 칸에는 반드시 `'승인' / '보류 유지' / '추가 수집 요청'` 중 하나만 사용합니다.

### 2. 모범 답안 — K1~K4 시각 대조 및 작성 서식 1

| 쌍 | 보고 A / B | 관측 차이 | 접수 차이 | 상충 유형 | 원인 후보(2개+) | 확인 방법 | 잠정 결론 |
|---|---|---|---|---|---|---|---|
| **K1** | C1(H-1, B4) / C3(I-1, A2) | 계산 불가 (C1 미기재) | **110분** (07:40→09:30) | 수량 (10여 대 vs 8개) | 시차(전달 지연 약 6시간), 구름·해상도 판독 한계 | H-1 재접촉 관측시각 확인, 재촬영 영상 대조 | 추가 수집 요청 |
| **K2** | K2A(I-2) / K2B(H-3) | '경' 표기로 계산 불가 | **35분** (11:05→11:40) | 존재 여부 (없음 vs 통과) | I-2 배터리 공백(10:35~10:55), H-3 '11:00경' 시차 | 드론 11:00 이후 연속 영상·H-3 정확한 통과 시각 재확인 | 보류 유지 |
| **K3** | C4(O-1, F5) / C3(I-1, A2) | 계산 불가 (C4 미확인) | **50분** (09:30→10:20) | 시각·위치 | 공개 사진 재사용(이전 게시물), 촬영 장소 상이 | 이미지 역검색 및 원본 메타데이터 대조 | 보류 유지 |
| **K4** | C2(S-1, A3) / K4B(H-2) | 계산 불가 (K4B 미기재) | **175분** (07:55→10:50) | 의미 (이동 준비 vs 정비) | 통신 내용 불명, H-2 전언 여부 미기재·검증 이력 없음 | 통신 분석 상세 확인, H-2 원출처 재확인 | 추가 수집 요청 |

### 3. 모범 답안 — 정보 공백 8개 이상(예시 외 5개+) & 표현 한계
- 공백 요인: ① `C1/K4B` 관측 시각 미기재(높음, K1·K4) ② `I-1` 구름·해상도 한계(중간, K1·K3) ③ `C4` 촬영 시각·장소 미확인 및 재사용 의심(높음, K3) ④ `C2` 통신 내용 불명(높음, K4·PIR) ⑤ `I-2` 배터리 교체 20분 공백 10:35~10:55(높음, K2) ⑥ `H-3` '11:00경' 부정확 시각(중간, K2) ⑦ `C1` 전달 지연 약 6시간(높음, K1·PIR) ⑧ `K4B` 재인용 원출처 미기재(중간, K4).
- 허용 표현: `'단일 출처 보고, 독립 확인 없음 — 잠정 등급 B3, 추가 수집 요청'` / 금지 표현: `'틀림없이 이동 중'`, `'통과한 것으로 보임'`.
- 판단 불가 3가지: 1) 적대 의도·정비/이동 여부(`K4`) 2) 재사용 게시물의 실제 촬영 시점(`K3`) 3) 관측 시각 미기재 보고의 현장 체류 시점(`K1`).

In [ ]:
# [교수자용 검증 코드] M3 K1~K4 접수 시각 분 단위 차이 계산 및 관측 차이 판정
import csv
from datetime import datetime

with open("02_데이터셋/보고_접수대장.csv", encoding="utf-8") as f:
    rows = {r["보고ID"]: r for r in csv.DictReader(f.readlines()[1:])}

pairs = [("K1", "C1", "C3"), ("K2", "K2A", "K2B"), ("K3", "C4", "C3"), ("K4", "C2", "K4B")]
for k_id, a_id, b_id in pairs:
    ta = datetime.strptime(rows[a_id]["접수시각"], "%H:%M")
    tb = datetime.strptime(rows[b_id]["접수시각"], "%H:%M")
    diff_min = int(abs((tb - ta).total_seconds()) // 60)
    oa, ob = rows[a_id]["관측시각"], rows[b_id]["관측시각"]
    obs_note = "계산 불가" if any(x in oa + ob for x in ["미기재", "미확인", "경", "구간"]) else "계산 가능"
    print(f"{k_id} ({a_id} vs {b_id}): 접수 차이 = {diff_min}분 | 관측 차이 = {obs_note} ({oa} vs {ob})")